# SysMLDocGen + OpenMBEE Interface Demo

这个 notebook 用来演示本系统的接口能力：登录、查看项目/模型/模板/文档、查看 OpenMBEE 适配接口目录，以及上传示例 SysML JSON 模型。

运行前请确认后端已经启动在 `http://127.0.0.1:8000`。

In [13]:
from pathlib import Path
import json
import requests
import pandas as pd

BASE_URL = "http://127.0.0.1:8000"
API_URL = f"{BASE_URL}/api"
WORKSPACE = Path(r"E:\sysml\sysml-docgen-system")
SAMPLE_MODEL = WORKSPACE / "docs" / "sample-sysml-model.json"

USERNAME = "admin"
PASSWORD = "123456"

session = requests.Session()

## 1. Health Check

In [14]:
resp = session.get(f"{BASE_URL}/health")
resp.raise_for_status()
resp.json()

{'status': 'ok', 'app': 'SysMLDocGen'}

## 2. Login

In [15]:
resp = session.post(f"{API_URL}/auth/login", json={"username": USERNAME, "password": PASSWORD})
resp.raise_for_status()
token_payload = resp.json()
session.headers.update({"Authorization": f"Bearer {token_payload['access_token']}"})
token_payload["user"]

{'id': 1,
 'username': 'admin',
 'email': 'admin@test.com',
 'full_name': '???',
 'role': 'admin',
 'is_active': True,
 'created_at': '2026-05-16T06:01:04'}

## 3. Core Data Overview

In [16]:
def get_json(path, **params):
    response = session.get(f"{API_URL}{path}", params={key: value for key, value in params.items() if value is not None})
    response.raise_for_status()
    return response.json()

projects = get_json("/projects")
models = get_json("/models")
templates = get_json("/documents/templates")
documents = get_json("/documents")

pd.DataFrame([
    {"name": "projects", "count": len(projects)},
    {"name": "models", "count": len(models)},
    {"name": "templates", "count": len(templates)},
    {"name": "documents", "count": len(documents)},
])

,name,count
0,projects,4
1,models,9
2,templates,5
3,documents,5


In [17]:
pd.DataFrame(projects)

,id,name,code,description,owner_id,created_at,updated_at
0,4,Version Test Project 005717,VERTEST-005717,automated versioning smoke test,1,2026-05-28T16:57:17,2026-05-28T16:57:17
1,3,????????-004634,VERTEST-004634,??????????,1,2026-05-28T16:46:34,2026-05-28T16:46:34
2,2,OMG SysML 标准模型测试,OMG-SYSML-001,测试,1,2026-05-16T06:39:10,2026-05-16T06:39:10
3,1,SysML ????????,DEMO-140343,????????????,1,2026-05-16T06:03:44,2026-05-16T06:03:44


## 4. OpenMBEE Adapter Endpoints

In [18]:
openmbee_config = get_json("/openmbee/config")
openmbee_config

{'mms_configured': False,
 'doc_convert_configured': False,
 'mms_url': None,
 'doc_convert_url': None}

In [19]:
openmbee_endpoints = get_json("/openmbee/mms/endpoints")
pd.DataFrame(openmbee_endpoints)

,name,method,path,description
0,mmsVersion,GET,/mmsversion,MMS 版本信息
1,authentication,POST,/authentication,用户认证
2,permissions,GET,/permissions,权限查询
3,orgs,GET,/orgs,组织列表
4,org,GET,/orgs/{orgId},组织详情
5,projects,GET,/projects?orgId={orgId},项目列表
6,project,GET,/projects/{projectId},项目详情
7,refs,GET,/projects/{projectId}/refs,分支/引用列表
8,ref,GET,/projects/{projectId}/refs/{refId},分支/引用详情
9,commits,GET,/projects/{projectId}/refs/{refId}/commits,提交历史


如果 `mms_configured` 是 `False`，说明还没有真实 MMS 服务。此时接口目录可以展示，但不能代理访问 MMS。

## 5. Upload Sample Model

In [20]:
if not projects:
    response = session.post(f"{API_URL}/projects", json={
        "name": "Jupyter Demo Project",
        "code": "JUPYTER-DEMO",
        "description": "Created from Jupyter notebook",
    })
    response.raise_for_status()
    projects = get_json("/projects")

project_id = projects[0]["id"]
project_id

4

In [21]:
with SAMPLE_MODEL.open("rb") as file_obj:
    response = session.post(
        f"{API_URL}/models/upload",
        data={"project_id": str(project_id), "name": "Jupyter Sample Model", "description": "Uploaded from notebook"},
        files={"file": (SAMPLE_MODEL.name, file_obj, "application/json")},
    )
response.raise_for_status()
uploaded_model = response.json()
uploaded_model

{'id': 10,
 'project_id': 4,
 'name': 'Jupyter Sample Model',
 'description': 'Uploaded from notebook',
 'source_filename': 'sample-sysml-model.json',
 'version': 2,
 'status': 'parsed',
 'uploaded_by': 1,
 'created_at': '2026-05-29T08:31:54'}

## 6. Inspect Model Graph

In [22]:
graph = get_json(f"/models/{uploaded_model['id']}/graph")
elements_df = pd.DataFrame(graph["elements"])
relations_df = pd.DataFrame(graph["relations"])
elements_df.head(20)

,id,model_id,element_uid,name,type,documentation,parent_uid
0,6235,10,REQ-001,自动生成工程文档,Requirement,系统应能够根据 SysML 模型和文档模板自动生成工程文档。,NaN
1,6236,10,REQ-002,模型数据可追溯,Requirement,生成文档应记录来源模型、模板和生成时间。,NaN
2,6237,10,BLK-001,文档生成系统,Block,系统核心块，负责模型管理、模板管理和文档生成。,NaN
3,6238,10,BLK-002,模型解析器,Block,解析 SysML/XMI/JSON 模型文件，抽取元素和关系。,BLK-001
4,6239,10,BLK-003,文档生成器,Block,根据模板渲染 HTML，并导出 DOCX/PDF。,BLK-001
5,6240,10,ACT-001,文档生成流程,Activity,选择项目、模型和模板后，系统生成文档并保存历史。,NaN
6,6241,10,REL-001,生成系统满足自动生成需求,Satisfy,NaN,NaN
7,6242,10,REL-002,文档生成器追踪模型数据,Trace,NaN,NaN


In [23]:
relations_df.head(20)

,id,model_id,source_uid,target_uid,relation_type,label
0,12381,10,BLK-001,REQ-001,Satisfy,生成系统满足自动生成需求
1,12382,10,BLK-003,REQ-002,Trace,文档生成器追踪模型数据
2,12383,10,BLK-001,BLK-002,containment,contains
3,12384,10,BLK-001,BLK-003,containment,contains


## 7. Generate Document With First Template

In [24]:
templates = get_json("/documents/templates", project_id=project_id)
if not templates:
    response = session.post(f"{API_URL}/documents/templates/default")
    response.raise_for_status()
    templates = get_json("/documents/templates", project_id=project_id)

template_id = templates[0]["id"]
response = session.post(f"{API_URL}/documents/generate", json={
    "project_id": project_id,
    "model_id": uploaded_model["id"],
    "template_id": template_id,
    "title": "Jupyter Generated SysML Document",
})
response.raise_for_status()
generated_doc = response.json()
generated_doc

{'id': 6,
 'project_id': 4,
 'model_id': 10,
 'template_id': 4,
 'title': 'Jupyter Generated SysML Document',
 'status': 'success',
 'html_content': '<h1>Jupyter Generated SysML Document</h1><p>Jupyter Sample Model</p>',
 'file_path': 'app\\generated\\document_6_4_10.html',
 'created_by': 1,
 'created_at': '2026-05-29T08:31:54'}